# Geometry of Truth Dataset — Exploratory Data Analysis

This notebook explores the Geometry of Truth dataset for use in the SAE-Faithful project. The goal is to inspect the dataset structure, understand the true/false statement labels, assess data quality, and prepare the dataset for later concept-detection and causal-faithfulness experiments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/saprmarks/geometry-of-truth/main/datasets/cities.csv"

cities = pd.read_csv(url)

cities.head()

In [ ]:
print("Shape:", cities.shape)

print("\nColumns:")
print(cities.columns.tolist())

print("\nData types:")
print(cities.dtypes)

print("\nMissing values:")
print(cities.isnull().sum())

print("\nLabel distribution:")
print(cities["label"].value_counts())

print("\nLabel proportions:")
print(cities["label"].value_counts(normalize=True))

In [ ]:
# Text length analysis
cities["char_length"] = cities["statement"].str.len()
cities["word_count"] = cities["statement"].str.split().str.len()

print("Character length summary:")
print(cities["char_length"].describe())

print("\nWord count summary:")
print(cities["word_count"].describe())

print("\nAverage word count by label:")
print(cities.groupby("label")["word_count"].mean())

In [ ]:
print("Duplicate rows:", cities.duplicated().sum())
print("Duplicate statements:", cities["statement"].duplicated().sum())

print("\nUnique cities:", cities["city"].nunique())
print("Unique stated countries:", cities["country"].nunique())
print("Unique correct countries:", cities["correct_country"].nunique())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label balance
cities["label"].value_counts().sort_index().plot(
    kind="bar",
    ax=axes[0]
)
axes[0].set_title("True vs False Statement Balance")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(["False (0)", "True (1)"], rotation=0)

# Word count distribution
cities["word_count"].plot(
    kind="hist",
    bins=8,
    ax=axes[1]
)
axes[1].set_title("Statement Word Count Distribution")
axes[1].set_xlabel("Number of Words")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

cities[cities["label"] == 0]["word_count"].plot(
    kind="hist",
    bins=8,
    alpha=0.6,
    label="False"
)

cities[cities["label"] == 1]["word_count"].plot(
    kind="hist",
    bins=8,
    alpha=0.6,
    label="True"
)

plt.title("Word Count Distribution by Truth Label")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
false_examples = cities[cities["label"] == 0].head(10)
true_examples = cities[cities["label"] == 1].head(10)

print("True examples:")
display(true_examples[["statement", "city", "country", "correct_country"]])

print("\nFalse examples:")
display(false_examples[["statement", "city", "country", "correct_country"]])

In [ ]:
city_counts = cities.groupby("city").size()

print("Statements per city:")
print(city_counts.value_counts().sort_index())

paired_check = cities.groupby("city")["label"].nunique()

print("\nCities with both true and false labels:")
print((paired_check == 2).sum())

print("Total unique cities:")
print(cities["city"].nunique())

In [ ]:
top_correct_countries = cities["correct_country"].value_counts().head(15)

print("Top 15 correct countries by number of cities:")
print(top_correct_countries)

plt.figure(figsize=(10, 5))
top_correct_countries.plot(kind="bar")
plt.title("Top 15 Correct Countries in the Geometry of Truth Cities Dataset")
plt.xlabel("Country")
plt.ylabel("Number of Cities")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
country_counts = cities["correct_country"].value_counts()

top_2_share = country_counts.head(2).sum() / country_counts.sum()
top_5_share = country_counts.head(5).sum() / country_counts.sum()

print(f"Share of cities from top 2 countries: {top_2_share:.2%}")
print(f"Share of cities from top 5 countries: {top_5_share:.2%}")

In [ ]:

summary = pd.DataFrame({
    "metric": [
        "rows",
        "original_columns",
        "derived_text_features",
        "missing_values",
        "duplicate_rows",
        "duplicate_statements",
        "true_statements",
        "false_statements",
        "unique_cities",
        "unique_correct_countries",
        "mean_word_count",
        "top_2_country_share",
        "top_5_country_share"
    ],
    "value": [
        len(cities),
        5,
        2,
        cities.isnull().sum().sum(),
        cities.duplicated().sum(),
        cities["statement"].duplicated().sum(),
        (cities["label"] == 1).sum(),
        (cities["label"] == 0).sum(),
        cities["city"].nunique(),
        cities["correct_country"].nunique(),
        cities["word_count"].mean(),
        top_2_share,
        top_5_share
    ]
})

summary

In [ ]:
summary.to_csv("../results/geometry_of_truth_eda_summary.csv", index=False)